In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
FD = pd.read_csv("C:\Users\ty\Downloads\Kene_projects\Fraud Detection Project\Kaggle Fraud Data for Testing\CSV\Wrangled\cleaned_fraud_data.csv")

In [3]:
# Transactions count per customer
FD = FD.sort_values(['customer_id', 'transaction_time'])
FD['txn_count_per_user'] = FD.groupby('customer_id')['transaction_id'].cumcount()

In [4]:
# Average amount per customer
FD['avg_amount_per_user'] = (
    FD.groupby('customer_id')['transaction_amount']
    .expanding()
    .mean()
    .shift(1)
    .reset_index(level=0, drop=True)
)

In [5]:
# Deviaton from user behavior
FD['amount_deviation'] = FD['transaction_amount'] - FD['avg_amount_per_user']

In [6]:
# Transaction frequency
FD = FD.sort_values(['customer_id', 'transaction_time'])
FD['time_diff'] = FD.groupby('customer_id')['transaction_time'].diff()
FD['txn_per_hour'] = 3600 / (FD['time_diff'] + 1e-6)

In [7]:
FD['time_diff'] = FD['time_diff'].fillna(0)
FD['txn_per_hour'] = FD['txn_per_hour'].replace([np.inf, -np.inf], 0)

In [8]:
# Transaction hour
FD['transaction_hour'] = (FD['transaction_time'] // 3600) % 24

In [9]:
# Flag high risk hours
FD['high_risk_hour'] = FD['transaction_hour'].apply(lambda x: 1 if x in [2,3,4] else 0)

In [10]:
# Log Transform Amount
FD['log_amount'] = np.log1p(FD['transaction_amount'])

In [11]:
# Relative amount
FD['relative_amount'] = FD['transaction_amount'] / (FD['avg_amount_per_user'] + 1)

In [13]:
FD.fillna({'avg_amount_per_user':0, 'amount_deviation':0, 'txn_per_hour':0, 'relative_amount':0}, inplace=True)

In [ ]:
FD.to_csv("C:\\Users\\ty\\Downloads\\Kene_projects\\Fraud Detection Project\\Kaggle Fraud Data for Testing\\CSV\\Wrangled\\feature_engineered_data.csv", index=False)